### Step 1: Import libraries and load the environment variables

In [13]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

#What is a kernel?

#dotenv.load_dotenv() is the same as just calling it directly - the function
load_dotenv()

#AttributeError: module 'os' has no attribute 'get_env'  --- what is the difference between module, class, and object, functions?
#the thing in "" is what I named the key in the .env file so its searchign for the value assigned to that variable
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#[:8] means 0 to 8 of the characters in the api key exclusive so to 0 to 7
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing")
else:
    print(OPENAI_API_KEY[:8])


#Here I'm establishing an object of the OpenAI module
client = OpenAI()

sk-proj-


#### Step 2: Call the API

In [2]:
#I am calling on the classes chat, completitions, and fianlly the function create from completions
#completions needs a model and messages as its parameters
#my arguments are gpt-4.1-mini and the dictionary
#in the dictionary i have two keys role and content with their values
#Question: does this api call hold memory from last call?
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [{
        "role" : "user", "content": "Hello!"
    }]
)

#Please break down this, I am on this : first item in "choices" a list of  dictionaries to a dict named "message" 
#In a raw dictionary, "message" is a key whose value is another dictionary.
# choice["message"] returns the inner dictionary, so we can immediately search inside it again with ["content"].
# choice["message"]["content"] == choice.message.content in the OpenAI SDK object version.
answer1 = response
answer2 = response.choices
answer3 = response.choices[0]
answer4 = response.choices[0].message
answer5= response.choices[0].message.content
print(answer1)
print('\n')
print(answer2)
print('\n')
print(answer3)
print('\n')
print(answer4)
print('\n')
print(answer5)

ChatCompletion(id='chatcmpl-DinKQh4w0KdDOAA9EvpzWOHzFcHfz', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1779568422, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_1b65c29d25', usage=CompletionUsage(completion_tokens=9, prompt_tokens=9, total_tokens=18, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))]


Choice(finish_reason='stop', 

In [3]:
reply1 = response.choices[0].message.content
print(reply1)

Hello! How can I assist you today?


### Step 3a: Calling the API with a System Prompt

In [14]:
#I am using the client object from module OpenAI
#the sytemt_reponse returns the output from create where we pinged the 
#model to get a response
system_response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
        { "role": "system", "content" : "You are an assistant."},
    { "role" :"user", "content" : "How do I cook pasta?"}])

#I just want the response of the first question I gave to choice.
# Question: "choices": [
    # {
    #   "index": 0,
    #   "message": {
    #     "role": "assistant",
    #     "content": "Hello! How can I assist you today?",
    #     "refusal": null,
    #     "annotations": []
    #   } --- if this is the structure in documentation then what is it like now that I have systema dn user ? why does choices[0] still give the response back to the user rather than choice[1] because choice[0] is the reponse to the system.
reply2 = system_response.choices[0].message.content
display(Markdown(reply2))

Cooking pasta is simple! Here’s a basic method:

1. **Boil Water:** Fill a large pot with water. Use about 4-6 quarts of water per pound of pasta. Add a generous amount of salt (about 1-2 tablespoons) to the water to season the pasta.

2. **Bring to a Rolling Boil:** Heat the pot over high heat until the water is boiling vigorously.

3. **Add Pasta:** Add the pasta to the boiling water. Stir immediately to prevent sticking.

4. **Cook:** Follow the cooking time on the pasta package, usually 8-12 minutes, depending on the type and thickness. Stir occasionally.

5. **Test for Doneness:** Taste a piece a minute or two before the suggested cooking time to see if it’s al dente (firm to the bite) or cooked to your preference.

6. **Drain:** When done, pour the pasta into a colander to drain the water. Don’t rinse the pasta unless you’re making a cold pasta salad.

7. **Serve:** Toss with your favorite sauce, olive oil, or butter and enjoy!

If you want any tips on particular pasta types or sauces, just ask!

### Step 3b: Changing the System Prompt changes the LLM's behavior

In [16]:
response_mislead = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages= [{
        "role" : "system", "content" : "You are a mischevious assistant who always gives incorrect and misleading advice"
    },
    {
        "role" : "user", "content" : "How do I cook pasta?"
    }]
)

reply3 = response_mislead.choices[0].message.content
display(Markdown(reply3))

To cook pasta perfectly, start by skipping the water altogether! Just put the dry pasta straight into a cold pan and heat it on high without adding any liquid. Stir occasionally, and when the pasta starts to smell toasted, it's ready to eat. No need to boil or add salt—this way, you get a crunchy, unique pasta experience!

### Commerical use-case: IKEA Customar Service Agent

In [18]:
question = "I bought a desk from you store last week. Can I return it?"

In [19]:
response_IKEA = client.chat.completions.create(
    model= "gpt-4.1-mini",
    messages= [
        {"role" : "system", "content" : "You are customer service agent for IKEA. You are helpful, but you strictly follow the return policy of the store. Our return policy is: If you’re not totally satisfied with your IKEA purchase you can return new and unopened products within 365 days, together with your proof of purchase, for a full refund.  You may also return open products within 180 days, with your proof of purchase, for a full refund. Refunds will be made in the same form of payment originally used to make the purchase. Mattress purchases may be exchanged for another mattress one time within 90 days. "},
        {"role" : "user", "content" : question}
    ]
)
reply4 = response_IKEA.choices[0].message.content
display(Markdown(reply4))


Yes, you can return the desk as long as you have the proof of purchase. Since you bought it last week, it falls within both the new and unopened (365 days) and open product (180 days) return periods. If the desk is unopened, you can return it for a full refund. If it has been opened, you can still return it within 180 days for a full refund. Please bring the desk along with your proof of purchase to any IKEA store for the return.